<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-5-Quantum-Structure-Hilbert-Born/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 5 — Quantum Structure
## Reviewer Walkthrough · Phase 22 Edition

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18604861.svg)](https://doi.org/10.5281/zenodo.18604861)

**Author:** E.S. Brooke · **Paper:** v2.1 main + v1.1 supplement · **Codebase:** v6.9

---

### What this notebook is

Paper 5 derives the quantum formalism — complex Hilbert, tensor product, Born rule, CPTP channels, decoherence — from admissibility. Supplies $\pi_Q$ to Paper 8.

### What this notebook is **not**

A full theory of quantum gravity ($L_{\rm QG\_P1\_closure}$ is UV completion as Type IV exit, not a full framework), and not a philosophical resolution of the measurement problem.

### Before you begin

If you are a cold AI agent or human reviewer new to APF, read these four files in `ai_context/` **first**:

1. **`ARGUMENT_FLOW.md`** — the one-page structural spine.
2. **`LOCAL_VS_IMPORTED.md`** — what this paper proves vs imports.
3. **`CLAIMS_LEDGER.md`** — row-by-row attack surface.
4. **`DO_NOT_CLAIM.md`** — predictable overclaims and how to avoid them.


## §1 · Setup

Clone the paper-companion repo and load the rendering helpers.

In [ ]:
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-5-Quantum-Structure-Hilbert-Born.git 2>/dev/null || true
%cd -q APF-Paper-5-Quantum-Structure-Hilbert-Born
!pip install -q -e . 2>&1 | tail -2

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from apf.apf_utils import dag_reset, dag_put, dag_has, dag_get
from fractions import Fraction
from IPython.display import display, Markdown, Latex, HTML
import inspect

print(f'Bank registry loaded: {len(REGISTRY)} checks.')
print('This repo: 6 bank-registered checks bundled.')

### Phase 22 `show()` helper

Every check returns a result dict. `show()` runs the check, badges the status colour-coded by epistemic tag, and surfaces the mathematical content inline.

Colour code: 🟢 `[P]` proved from A1 · 🟡 `[P_structural]` conditional on upstream · 🟣 `[P_arith]` arithmetic identity · 🔵 `[P+lattice]` lattice QCD input · 🟠 `[C]` conjecture · 🔴 `[FAIL]` check did not pass.

In [ ]:

def _epistemic_badge(tag):
    colors = {
        'P': ('🟢', '#2ecc71', 'proved from A1'),
        'P_structural': ('🟡', '#f1c40f', 'conditional on upstream derivation'),
        'P_arith': ('🟣', '#9b59b6', 'arithmetic identity once formula chosen'),
        'P+lattice': ('🔵', '#3498db', 'proved using lattice QCD input'),
        'C': ('🟠', '#e67e22', 'conjecture; open, flagged'),
    }
    emoji, col, explain = colors.get(str(tag).strip('[]'), ('⚪', '#7f8c8d', 'unknown'))
    return f'<span style="background:{col}22;border:1px solid {col};border-radius:4px;padding:2px 8px;color:{col};font-weight:600">{emoji} {tag}</span> <em style="color:#666;font-size:0.9em">{explain}</em>'


def _render_value(v):
    if isinstance(v, Fraction):
        return f'$\\displaystyle \\frac{{{v.numerator}}}{{{v.denominator}}} = {float(v):.6f}$'
    if isinstance(v, dict) and all(isinstance(val, Fraction) for val in v.values()):
        items = [f'{k}={frac.numerator}/{frac.denominator}' for k, frac in v.items()]
        return '$' + ',\\ '.join(items) + '$'
    if isinstance(v, float):
        return f'`{v:.9g}`'
    if isinstance(v, (list, tuple)) and len(v) < 8:
        return '`' + ', '.join(str(x) for x in v) + '`'
    return f'`{v}`'


def show(check_name, *, run=True, verbose=True):
    try:
        check = get_check(check_name)
    except KeyError:
        display(Markdown(f'**❓ Check `{check_name}` not found.**'))
        return None

    display(Markdown(f'#### `{check_name}`'))
    doc = (check.__doc__ or '').strip()
    first_line = doc.split('\n')[0] if doc else '(no docstring)'
    display(Markdown(f'**Statement:** {first_line}'))

    if not run:
        return None

    try:
        result = check()
        passed = True
    except Exception as e:
        result = {'error': f'{type(e).__name__}: {e}'}
        passed = False

    tag = 'P'
    if isinstance(result, dict):
        for k in ('epistemic_status', 'epistemic', 'tag'):
            if k in result:
                tag = result[k]
                break
    if not passed:
        display(Markdown(f'**Status:** <span style="color:#e74c3c;font-weight:700">🔴 [FAIL]</span>'))
    else:
        display(Markdown(f'**Status:** {_epistemic_badge(tag)}'))

    if isinstance(result, dict):
        if 'key_result' in result:
            display(Markdown(f'**Key result:** {_render_value(result["key_result"])}'))
        if 'error' in result:
            display(Markdown(f'**Error:** `{result["error"]}`'))

        if verbose:
            skip = {'key_result', 'name', 'epistemic', 'epistemic_status', 'tag',
                    'dependencies', 'cross_refs', 'error', 'artifacts', 'statement',
                    'identity', 'consistent'}
            extra = {k: v for k, v in result.items() if k not in skip}
            if extra:
                rows = []
                for k, v in list(extra.items())[:10]:
                    rows.append(f'| `{k}` | {_render_value(v)} |')
                if rows:
                    display(Markdown('**Fields surfaced by the check:**\n\n| Field | Value |\n|---|---|\n' + '\n'.join(rows)))

        if 'dependencies' in result and result['dependencies']:
            deps = result['dependencies']
            if isinstance(deps, (list, tuple)):
                deps_str = ' · '.join(f'`{d}`' for d in deps)
                display(Markdown(f'**Depends on:** {deps_str}'))

    return result


print('show() helper loaded. Phase 22 gorgeous-math rendering enabled.')


## §2 · Linearity from coherence closure

Closed admissibility forces linear dynamics. Non-linear evolution violates coherence closure.

## §3 · Complex Hilbert via Frobenius trichotomy

$$\text{Admissibility closure} \;+\; L_{\rm loc}\text{ commutativity} \;\Rightarrow\; \underbrace{\mathbb{R}\;\xmark\; \mathbb{C}\;\checkmark\; \mathbb{H}\;\xmark}_{\text{Frobenius division algebras}}$$

- $\mathbb{R}$ ruled out by R4 (no non-trivial irreps).
- $\mathbb{H}$ ruled out by $L_{\rm loc}$ commutativity.
- **Only $\mathbb{C}$ survives.**

In [ ]:
show('check_T_Hilbert_complex')

## §4 · Born rule via Gleason

$$\boxed{\quad P(\psi \to \phi) \;=\; |\langle \phi | \psi \rangle|^2 \quad}$$

Derived via Gleason's theorem applied to the lattice of admissible projections.

In [ ]:
show('check_T_Born')

## §5 · Tensor product + CPTP + decoherence

- **$T_{\rm tensor}$** — composite systems obey tensor-product structure.
- **$T_{\rm CPTP}$** — channels are CPTP (via Stinespring).
- **$T_{\rm decoherence}$** — Type III regime exit via record-locking channels.

In [ ]:
show('check_T_tensor')

In [ ]:
show('check_T_CPTP')

In [ ]:
show('check_T_decoherence')

## §6 · Spectral action internalisation

**$L_{\rm spectral\_action\_internal}$** — the spectral action at Boltzmann cutoff equals the APF partition function. Seeley-DeWitt coefficients:

$$a_0 = 61,\qquad a_2 \approx 21.985,\qquad a_4 \approx 87.201$$

**$a_0 = 61 = C_{\rm total}$** — triple reading (Seeley-DeWitt / $\pi_F$ / $\pi_C$). See Paper 7 for full triple-reading treatment.

In [ ]:
show('check_L_spectral_action_internal')

## §7 · Full bank pass

Run every check in this repo's bundled codebase subset.

In [ ]:
from apf.bank import run_all
from collections import Counter
results = run_all()
outcomes = Counter()
for name, res in results.items():
    if isinstance(res, dict) and 'error' in res:
        outcomes['ERROR'] += 1
    else:
        outcomes['PASS'] += 1
print(f'Total: {len(results)} checks')
for status, n in outcomes.most_common():
    print(f'  {status}: {n}')

### Where to go next

- **Paper PDF** — the main paper + Technical Supplement in this repo.
- **`ai_context/`** — the four audit-native files (ARGUMENT_FLOW, LOCAL_VS_IMPORTED, CLAIMS_LEDGER, DO_NOT_CLAIM).
- **[Canonical codebase v6.9](https://doi.org/10.5281/zenodo.18604548)** — the full 420-theorem bank.
- **[Paper 8 companion repo](https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger)** — the pilot implementation of Phase 22 (anti-smuggling tests + minimal working example + full gorgeous-math Colab).

### Citation

```bibtex
@software{Brooke_Paper5_2026,
  author  = {Brooke, Ethan S.},
  title   = {Quantum Structure from Finite Enforceability},
  year    = 2026,
  version = {v2.1 main + v1.1 supplement},
  doi     = {10.5281/zenodo.18604861}
}
```

*Paper-companion repo · v2.1 main + v1.1 supplement · Phase 22 gorgeous-math edition · 2026-04-24.*